# Fetching more data to help improve the accuracy of the model

We already walked through the steps before, so we will quickly repeat them below

In [15]:
import pandas as pd

In [16]:
df_abc = pd.read_csv("data/news/original/abcnews-date-text.csv")

In [17]:
df_abc.head()

,publish_date,headline_text
0,20030219,aba decides against community broadcasting lic...
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers


Filtering the dataset

In [18]:
df_abc.dropna(inplace=True)
df_abc.reset_index(drop=True, inplace=True)

df_abc["date"] = pd.to_datetime(df_abc["publish_date"].astype(str), format="%Y%m%d", errors="coerce")
df_abc.dropna(inplace=True) # remove rows where date conversion failed
df_abc.reset_index(drop=True, inplace=True)

min_date_abc = df_abc["date"].min()
max_date_abc = df_abc["date"].max()

df_abc["summary"] = (
    df_abc["headline_text"].fillna("").astype(str).str.strip()
).str.strip()

df_abc["source"] = "abcnews"

df_abc[["date", "summary", "source"]].to_csv(
    "data/news/filtered/abcnews_filtered.csv", index=False
)


In [22]:
df_cnbc_filtered = pd.read_csv(
    "data/news/filtered/cnbc_filtered.csv", parse_dates=["date"]
)
df_guardian_filtered = pd.read_csv(
    "data/news/filtered/guardian_filtered.csv", parse_dates=["date"]
)
df_reuters_filtered = pd.read_csv(
    "data/news/filtered/reuters_filtered.csv", parse_dates=["date"]
)
df_abcnews_filtered = pd.read_csv(
    "data/news/filtered/abcnews_filtered.csv", parse_dates=["date"]
)

df_all = pd.concat(
    [df_cnbc_filtered, df_guardian_filtered, df_reuters_filtered, df_abcnews_filtered], ignore_index=True
)

In [24]:
df_all.to_csv("data/news/filtered/all_filtered_optimization.csv", index=False)
df_all.head()

,date,summary,source,adj_date
0,2020-07-17 19:51:00,Jim Cramer: A better way to invest in the Covi...,cnbc,2020-07-20
1,2020-07-17 19:33:00,Cramer's lightning round: I would own Teradyne...,cnbc,2020-07-20
2,2020-07-17 19:25:00,"Cramer's week ahead: Big week for earnings, ev...",cnbc,2020-07-20
3,2020-07-17 16:24:00,IQ Capital CEO Keith Bliss says tech and healt...,cnbc,2020-07-20
4,2020-07-16 19:36:00,Wall Street delivered the 'kind of pullback I'...,cnbc,2020-07-17


In [19]:
%%capture
%pip install nltk

import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")  # for lemmatization

In [20]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

finance_whitelist = {
    "up",
    "down",
    "above",
    "below",
    "under",
    "over",
    "rise",
    "fall",
    "not",
    "no",
    "nor",
    "neither",
    "never",
    "none",
    "more",
    "most",
    "few",
    "less",
}

stop_words.difference_update(finance_whitelist)

In [21]:
def clean_text(text):
    # 1. Lowercase all text
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # 3. Tokenize using Punkt
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # 5. Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [23]:
# finding all the times that are past 4pm or 16 in military time and increasing the date by 1 day unless it is friday
df_all["adj_date"] = df_all["date"]

# Saturday posts get moved +2 days (to Monday)
df_all.loc[df_all["adj_date"].dt.weekday == 5, "adj_date"] += pd.Timedelta(days=2)

# Sunday posts get moved +1 day (to Monday)
df_all.loc[df_all["adj_date"].dt.weekday == 6, "adj_date"] += pd.Timedelta(days=1)

# Friday posts after 4pm get moved +3 days (to Monday)
friday_after_4 = (df_all["date"].dt.weekday == 4) & (  # Friday
    df_all["date"].dt.hour >= 16
)  # After 4pm
df_all.loc[friday_after_4, "adj_date"] += pd.Timedelta(days=3)

# Weekday posts after 4pm get moved +1 day
after_4 = (df_all["date"].dt.weekday != 4) & (  # Not Friday
    df_all["date"].dt.hour >= 16
)  # After 4pm
df_all.loc[after_4, "adj_date"] += pd.Timedelta(days=1)

df_all["adj_date"] = df_all["adj_date"].dt.normalize()

In [25]:
df_news = pd.read_csv("data/news/filtered/all_filtered_optimization.csv")

df_news["clean_summary"] = df_news["summary"].apply(clean_text)

df_news.head()

,date,summary,source,adj_date,clean_summary
0,2020-07-17 19:51:00,Jim Cramer: A better way to invest in the Covi...,cnbc,2020-07-20,jim cramer better way invest covid19 vaccine g...
1,2020-07-17 19:33:00,Cramer's lightning round: I would own Teradyne...,cnbc,2020-07-20,cramers lightning round would teradyne mad mon...
2,2020-07-17 19:25:00,"Cramer's week ahead: Big week for earnings, ev...",cnbc,2020-07-20,cramers week ahead big week earnings even bigg...
3,2020-07-17 16:24:00,IQ Capital CEO Keith Bliss says tech and healt...,cnbc,2020-07-20,iq capital ceo keith bliss say tech healthcare...
4,2020-07-16 19:36:00,Wall Street delivered the 'kind of pullback I'...,cnbc,2020-07-17,wall street delivered kind pullback ive waitin...


In [26]:
df_news_clean = df_news[["adj_date", "clean_summary"]].copy()

df_news_clean.rename(
    columns={"adj_date": "date", "clean_summary": "summary"}, inplace=True
)

df_news_grouped = df_news_clean.groupby("date")["summary"].apply(" ".join).reset_index()
df_news_grouped.to_csv("data/news/cleaned/all_clean_optimization.csv", index=False)

df_news_grouped.head()

,date,summary
0,2003-02-19,aba decides community broadcasting licence act...
1,2003-02-20,15 dead rebel bombing raid philippine army aba...
2,2003-02-21,accc timid petrol price investigation action w...
3,2003-02-24,86 confirmed dead u nightclub fire act tourist...
4,2003-02-25,4 million pay sacked ceo aid organisation disa...
